In [1]:
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
import re
import tkinter as tk
from tkinter import messagebox
from nltk.tokenize import word_tokenize

# nltk.download('stopwords')

In [2]:
df = pd.read_excel('TABEL DATA LATIH HATESPEECH RISET.xlsx')

In [3]:
df.columns = ['username', 'pesan', 'label']

In [4]:
df.head()

,username,pesan,label
0,__succiduous,Udah jelek brengsek pula,Ras
1,KemenagMempawah,Lucunya penghuni negeri ini selalu di hiasi da...,Agama
2,1stKOREANguy1,"Yang jelek + miskin udah pasti bukan Kristen ,...",Ras
3,newsutdofficial,@MurtadhaOne1 Mereka memanfaatkan kebodohan ka...,Agama
4,pshycosocial_,orang jelek kalo tarik tambang pasti kalah ter...,Ras


In [5]:
df.isnull().sum().sum()

np.int64(11)

In [6]:
df.duplicated().sum()

np.int64(45)

In [7]:
df = df.drop_duplicates(keep='first')
df = df.dropna()

### Cleaning etc

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1955 entries, 0 to 2002
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   username  1955 non-null   object
 1   pesan     1955 non-null   object
 2   label     1955 non-null   object
dtypes: object(3)
memory usage: 61.1+ KB


In [9]:
df['pesan'] = df['pesan'].astype(str)

In [10]:


# def cleaning(text):
#     stopword = set(stopwords.words('indonesian'))
#     faktor = StemmerFactory()
#     Stemmer = faktor.create_stemmer()
    
#     text_bersih = text.lower() # case normalization
#     text_bersih = re.sub(r'https\S+', '', text_bersih) # text cleaning
#     text_bersih = re.sub(r'@[A-Za-z0-9_]+', '', text_bersih)
#     text_bersih = re.sub(r'#([A-Za-z0-9_])+', '', text_bersih)
#     text_bersih = re.sub(r'[^a-z\s]', '', text_bersih)
#     text_bersih = re.sub(r'\d+', '', text_bersih)
#     text_bersih = re.sub(r'\s+', ' ', text_bersih)
#     text_bersih = Stemmer.stem(text_bersih)
    
#     return text_bersih


# # ts = 'woi prabowok 948291 *&*@() amb mana sawit graTISNYA wok'

# # hasil = cleaning(ts)
# # print(hasil)

# df['pesan_bersih'] = df['pesan'].apply(cleaning)

In [11]:
df.head()

,username,pesan,label
0,__succiduous,Udah jelek brengsek pula,Ras
1,KemenagMempawah,Lucunya penghuni negeri ini selalu di hiasi da...,Agama
2,1stKOREANguy1,"Yang jelek + miskin udah pasti bukan Kristen ,...",Ras
3,newsutdofficial,@MurtadhaOne1 Mereka memanfaatkan kebodohan ka...,Agama
4,pshycosocial_,orang jelek kalo tarik tambang pasti kalah ter...,Ras


In [12]:
df['label'] = df['label'].map({'Ras':0,
                               'Agama':1,
                               'Netral':2})

In [13]:
# tf id f



## EDA

## MODEL

In [14]:
df.head()

,username,pesan,label
0,__succiduous,Udah jelek brengsek pula,0
1,KemenagMempawah,Lucunya penghuni negeri ini selalu di hiasi da...,1
2,1stKOREANguy1,"Yang jelek + miskin udah pasti bukan Kristen ,...",0
3,newsutdofficial,@MurtadhaOne1 Mereka memanfaatkan kebodohan ka...,1
4,pshycosocial_,orang jelek kalo tarik tambang pasti kalah ter...,0


In [15]:
X = df['pesan_bersih'].values
y = df['label'].values

KeyError: 'pesan_bersih'

In [ ]:
y.shape

(1955,)

In [ ]:
idx_0 = np.where(y == 0)[0]
idx_1 = np.where(y == 1)[0]

np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

train_0 = int(0.8 * len(idx_0))
train_1 = int(0.8 * len(idx_1))

train_idx = np.concatenate((idx_0[:train_0], idx_1[:train_1]))
test_idx = np.concatenate((idx_0[train_0:], idx_1[train_1:]))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [ ]:
df['label'].unique()

array([0, 1, 2])

In [ ]:
all_words = set()
for doc in X_train:
    all_words.update(doc.split())

word_list = list(all_words)
word_to_idx = {word: idx for idx, word in enumerate(word_list)}

num_docs = len(X_train)
vocab_size = len(word_list)

# traainkl
tf_matrix = np.zeros((num_docs, vocab_size))
for i, doc in enumerate(X_train):
    words = doc.split()
    word_counts = np.bincount([word_to_idx[word] for word in words if word in word_to_idx], minlength=vocab_size)
    tf_matrix[i] = word_counts / len(words) if len(words) > 0 else 0

doc_freq = np.sum(tf_matrix > 0, axis=0)
idf = np.log(num_docs / (1 + doc_freq))

tfidf_train = tf_matrix * idf

# tes
tfidf_test = np.zeros((len(X_test), vocab_size))
for i, doc in enumerate(X_test):
    words = doc.split()
    word_counts = np.bincount([word_to_idx[word] for word in words if word in word_to_idx], minlength=vocab_size)
    tfidf_test[i] = (word_counts / len(words) if len(words) > 0 else 0) * idf

print(f'TF-IDF Train shape: {tfidf_train.shape}')
print(f'TF-IDF Test shape: {tfidf_test.shape}')

TF-IDF Train shape: (907, 3130)
TF-IDF Test shape: (228, 3130)


In [ ]:
classes = np.unique(y_train)
classes

array([0, 1])

In [ ]:
class_data = {}

for i in classes:
    Xc = X_train[y_train == i]
    class_data[i] = Xc

print(f'Class 0 {class_data[0].shape[0]}')
print(f'Class 1 {class_data[1].shape[0]}')

Class 0 471
Class 1 436


In [ ]:
total_data = X.shape[0]
prior = {}

for c in class_data:
    prior[c] = class_data[c].shape[0] / total_data

for c in prior:
    print(f'prior {c}, {prior[c]:2f}')

prior 0, 0.240921
prior 1, 0.223018


In [ ]:
df.head()

,username,pesan,label,pesan_bersih
0,__succiduous,Udah jelek brengsek pula,0,udah jelek brengsek pula
1,KemenagMempawah,Lucunya penghuni negeri ini selalu di hiasi da...,1,lucu huni negeri ini selalu di hias dan warna ...
2,1stKOREANguy1,"Yang jelek + miskin udah pasti bukan Kristen ,...",0,yang jelek miskin udah pasti bukan kristen tap...
3,newsutdofficial,@MurtadhaOne1 Mereka memanfaatkan kebodohan ka...,1,mereka manfaat bodoh kadrun lalu berita hoax
4,pshycosocial_,orang jelek kalo tarik tambang pasti kalah ter...,0,orang jelek kalo tarik tambang pasti kalah ter...


In [ ]:
text = df['pesan_bersih'].columns

def build_vocabb(feature):
    return list(feature)

vocab = build_vocabb(text)

print(f'vocab {vocab}')
print('jumlah vocab', len(vocab))

AttributeError: 'Series' object has no attribute 'columns'

In [ ]:
fitur = len(vocab)
likehood = {}

for c in classes:
    Xc = X_train[y_train == c]
    word = Xc.sum(axis=0)
    total = word.sum()

    likehood[c] = (word + 1) / (total + 1 * fitur)

AttributeError: 'str' object has no attribute 'sum'

In [ ]:
def prediksi(x):
    posterior = {}

    for c in classes:
        log1 = np.log(prior[c])
        log2 = np.sum(x * np.log(likehood[c]))
        posterior[c] = log1 + log2
    return max(posterior, key=posterior.get)

In [ ]:
y_pred = prediksi(tfidf_train)

accuracy = np.mean(y_pred == y_test)
print(f'Akurasi {accuracy}')

KeyError: np.int64(0)

## EVALUASI

In [ ]:
root = tk.Tk()
root.geometry('500x1000')
root.configure(bg="#414141")
root.title('Klasifikasi Sentimen Hatespeech')

frame = tk.Frame(root,
                 bg='#414141')

frame.place(relx=0.5,
            rely=0.35,
            anchor='center')

# label1
label1 = tk.Label(frame,
                  text='Masukan Kalimat',
                  font=('Arial', 28, 'bold'),
                  bg='#414141',
                  fg='#FFFFFF')
label1.pack()

# entry1
entry1 = tk.Entry(frame,
                  font=('Arial', 20),
                  bg='#868686',
                  fg='#FFFFFF')
entry1.pack(pady=15)

# btn1
btn1 = tk.Button(frame,
                 text='Cek HateSpeech',
                 bg="#7D7B7B",
                 fg='#FFFFFF')
btn1.pack()

root.mainloop()